In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
from IPython.display import display


# =========================
# Option B (Approximation)
# =========================
# Uses already-saved net returns + ADT (Avg Daily Turnover) to approximate
# net returns under alternative transaction-cost assumptions.
# No files are written in this notebook.

TC_GRID_BPS = [0, 5, 10, 20, 50]
BASE_TC_BPS = 2.0  # current training/evaluation baseline in the project
MARKETS = ["QQQ", "NKY", "EUSTX"]
SIGNIFICANT_DIGITS = 6


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "Results_Daily").exists() and (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root from current notebook location.")


def equity_to_returns(tab: np.ndarray) -> np.ndarray:
    tab = np.array(tab, dtype=np.float64)
    return (tab[1:] / tab[:-1]) - 1


def absolute_return(tab: np.ndarray) -> float:
    ret = equity_to_returns(tab)
    if len(ret) == 0:
        return 0.0
    return float((np.prod(1 + ret) - 1.0) * 100.0)


def arc_pct(tab: np.ndarray, annualization: int = 252) -> float:
    tab = np.array(tab, dtype=np.float64)
    ret = equity_to_returns(tab)
    length = len(tab)
    if length <= 1:
        return 0.0
    a_rtn = np.prod(1 + ret[:-1]) if len(ret) > 1 else 1.0
    if a_rtn <= 0:
        return 0.0
    return float(100.0 * (math.pow(a_rtn, annualization / length) - 1.0))


def max_drawdown_pct(tab: np.ndarray) -> float:
    ret = equity_to_returns(tab)
    if len(ret) == 0:
        return 0.0
    cum_returns = np.cumprod(1 + ret)
    cum_max = np.maximum.accumulate(cum_returns)
    drawdowns = (cum_max - cum_returns) / cum_max
    return float(np.max(drawdowns) * 100.0)


def asd_pct(tab: np.ndarray, annualization: int = 252) -> float:
    ret = equity_to_returns(tab)
    if len(ret) == 0:
        return 0.0
    return float(np.sqrt(annualization) * np.std(ret) * 100.0)


def sgn(x: float) -> int:
    if x == 0:
        return 0
    return int(abs(x) / x)


def ir2(tab: np.ndarray, annualization: int = 252) -> float:
    asd = asd_pct(tab, annualization)
    arc = arc_pct(tab, annualization)
    mdd = max_drawdown_pct(tab)
    denom = asd * mdd
    if denom == 0:
        return 0.0
    numer = (arc ** 2) * sgn(arc)
    return float(max(numer / denom, 0.0))


def sharpe_ann(returns: np.ndarray, annualization: int = 252) -> float:
    r = np.asarray(returns, dtype=float)
    if len(r) <= 1 or np.std(r) <= 1e-12:
        return 0.0
    return float(np.mean(r) / np.std(r) * np.sqrt(annualization))


def compute_metrics_from_returns(daily_returns: np.ndarray, annualization: int = 252) -> dict:
    r = np.asarray(daily_returns, dtype=float)
    eq = np.concatenate(([1.0], np.cumprod(1.0 + r)))
    return {
        "Absolute Return (%)": absolute_return(eq),
        "ARC (%)": arc_pct(eq, annualization),
        "ASD (%)": asd_pct(eq, annualization),
        "Max Drawdown (%)": max_drawdown_pct(eq),
        "IR2": ir2(eq, annualization),
        "Sharpe": sharpe_ann(r, annualization),
        "N Days": int(len(r)),
    }


def safe_break_even(mu_diff: float, adt_frac: float, tc_base_bps: float) -> float:
    # r_new = r_base - adt_frac * ((tc_new - tc_base)/10000)
    # mean(r_new - benchmark) = 0 => tc_new = tc_base + mean_diff / (adt_frac/10000)
    if adt_frac <= 1e-12:
        return np.nan
    return float(tc_base_bps + mu_diff / (adt_frac / 10000.0))


repo_root = find_repo_root(Path.cwd().resolve())
results_root = repo_root / "Results_Daily"

run_rows = []
grid_rows = []
break_even_rows = []

for market in MARKETS:
    market_dir = results_root / market
    if not market_dir.exists():
        continue

    run_dirs = sorted([p for p in market_dir.iterdir() if p.is_dir()])
    for run_dir in run_dirs:
        returns_path = run_dir / "rl_daily_returns_oos.csv"
        perf_path = run_dir / "rl_performance_metrics.csv"

        if not returns_path.exists() or not perf_path.exists():
            continue

        ret_df = pd.read_csv(returns_path)
        if "RL Agent" not in ret_df.columns:
            continue

        benchmark_col = "QQQ" if "QQQ" in ret_df.columns else None
        if benchmark_col is None:
            continue

        perf_df = pd.read_csv(perf_path)
        if "Unnamed: 0" in perf_df.columns:
            perf_df = perf_df.rename(columns={"Unnamed: 0": "Strategy"})

        rl_row = perf_df.loc[perf_df["Strategy"] == "RL Agent"]
        if rl_row.empty or "Avg Daily Turnover (%)" not in rl_row.columns:
            continue

        adt_pct = float(rl_row["Avg Daily Turnover (%)"].iloc[0])
        adt_frac = adt_pct / 100.0

        r_base = ret_df["RL Agent"].to_numpy(dtype=float)
        r_bench = ret_df[benchmark_col].to_numpy(dtype=float)

        mu_base = float(np.mean(r_base))
        mu_bench = float(np.mean(r_bench))
        mu_diff = mu_base - mu_bench

        run_label = run_dir.name

        run_rows.append(
            {
                "Market": market,
                "Run": run_label,
                "N": len(r_base),
                "ADT (%)": adt_pct,
                "Base TC (bps)": BASE_TC_BPS,
                "Mean RL net daily": mu_base,
                "Mean Benchmark daily": mu_bench,
                "Mean Daily Alpha (RL-Bench)": mu_diff,
            }
        )

        tc_be_zero = safe_break_even(mu_base, adt_frac, BASE_TC_BPS)
        tc_be_vs_bench = safe_break_even(mu_diff, adt_frac, BASE_TC_BPS)

        break_even_rows.append(
            {
                "Market": market,
                "Run": run_label,
                "ADT (%)": adt_pct,
                "Break-even TC vs 0 return (bps)": tc_be_zero,
                "Break-even TC vs benchmark mean (bps)": tc_be_vs_bench,
            }
        )

        for tc_bps in TC_GRID_BPS:
            tc_shift = (tc_bps - BASE_TC_BPS) / 10000.0
            r_approx = r_base - adt_frac * tc_shift
            m = compute_metrics_from_returns(r_approx, annualization=252)

            grid_rows.append(
                {
                    "Market": market,
                    "Run": run_label,
                    "TC (bps)": tc_bps,
                    "Approx Mean Daily Return": float(np.mean(r_approx)),
                    **m,
                }
            )

run_summary = pd.DataFrame(run_rows).sort_values(["Market", "Run"]).reset_index(drop=True)
grid_results = pd.DataFrame(grid_rows).sort_values(["Market", "Run", "TC (bps)"]).reset_index(drop=True)
break_even = pd.DataFrame(break_even_rows).sort_values(["Market", "Run"]).reset_index(drop=True)

pd.set_option("display.float_format", lambda x: f"{x:.{SIGNIFICANT_DIGITS}f}")

print("Option B TC Sensitivity Assessment (Approximate) — no files saved")
print(f"Repo: {repo_root}")
print(f"Markets found: {', '.join(sorted(run_summary['Market'].unique())) if not run_summary.empty else 'None'}")
print(f"Runs analyzed: {len(run_summary)}")
print(f"TC grid (bps): {TC_GRID_BPS}")
print(f"Base TC assumed (bps): {BASE_TC_BPS}")

print("\nRun-level inputs used")
display(run_summary)

print("\nBreak-even TC estimates")
display(break_even)

print("\nApproximate metrics under TC grid")
display(grid_results)

# Compact thesis-ready comparison views
arc_pivot = (
    grid_results
    .pivot_table(index=["Market", "Run"], columns="TC (bps)", values="ARC (%)")
    .reindex(columns=TC_GRID_BPS)
)
ir2_pivot = (
    grid_results
    .pivot_table(index=["Market", "Run"], columns="TC (bps)", values="IR2")
    .reindex(columns=TC_GRID_BPS)
)
sharpe_pivot = (
    grid_results
    .pivot_table(index=["Market", "Run"], columns="TC (bps)", values="Sharpe")
    .reindex(columns=TC_GRID_BPS)
)

print("\nCompact ARC (%) by TC (bps)")
display(arc_pivot)

print("\nCompact IR2 by TC (bps)")
display(ir2_pivot)

print("\nCompact Sharpe by TC (bps)")
display(sharpe_pivot)

break_even_compact = (
    break_even[[
        "Market",
        "Run",
        "Break-even TC vs 0 return (bps)",
        "Break-even TC vs benchmark mean (bps)",
    ]]
    .set_index(["Market", "Run"])
)

print("\nBreak-even TC summary (bps)")
display(break_even_compact)

print("\nApproximation note:")
print("- Net returns are shifted by ADT-based linear TC adjustment; daily turnover path is not available in saved RL outputs.")
print("- Use this for sensitivity direction and rough break-even levels, not as exact pathwise recomputation.")

Option B TC Sensitivity Assessment (Approximate) — no files saved
Repo: /Users/kamilkashif/Documents/University/Masters Thesis/MS-Thesis-Deep-RL-KK
Markets found: EUSTX, NKY, QQQ
Runs analyzed: 15
TC grid (bps): [0, 5, 10, 20, 50]
Base TC assumed (bps): 2.0

Run-level inputs used


,Market,Run,N,ADT (%),Base TC (bps),Mean RL net daily,Mean Benchmark daily,Mean Daily Alpha (RL-Bench)
0,EUSTX,EUSTX_LSTM_DAILY_1,4086,10.613400,2.000000,0.000394,0.000397,-0.000003
1,EUSTX,EUSTX_LSTM_DAILY_2,4086,11.415800,2.000000,0.000368,0.000397,-0.000029
2,EUSTX,EUSTX_LSTM_NC_DAILY,4086,15.661300,2.000000,0.000354,0.000397,-0.000043
3,EUSTX,EUSTX_LSTM_NC_DAILY_2,4086,12.696500,2.000000,0.000403,0.000397,0.000006
4,EUSTX,EUSTX_TRANSFORMERS_DAILY_1,4086,8.418200,2.000000,0.000400,0.000397,0.000003
5,NKY,NKY_LSTM_DAILY_1,3917,23.496800,2.000000,0.000508,0.000462,0.000046
6,NKY,NKY_LSTM_DAILY_2,3917,18.858300,2.000000,0.000412,0.000462,-0.000050
7,NKY,NKY_LSTM_NC_DAILY,3917,20.789000,2.000000,0.000461,0.000462,-0.000001
8,NKY,NKY_LSTM_NC_DAILY_2,3917,20.035700,2.000000,0.000461,0.000462,-0.000001
9,NKY,NKY_TRANSFORMERS_DAILY_1,3917,20.650300,2.000000,0.000435,0.000462,-0.000026



Break-even TC estimates


,Market,Run,ADT (%),Break-even TC vs 0 return (bps),Break-even TC vs benchmark mean (bps)
0,EUSTX,EUSTX_LSTM_DAILY_1,10.613400,39.075886,1.672892
1,EUSTX,EUSTX_LSTM_DAILY_2,11.415800,34.229138,-0.544854
2,EUSTX,EUSTX_LSTM_NC_DAILY,15.661300,24.615513,-0.731868
3,EUSTX,EUSTX_LSTM_NC_DAILY_2,12.696500,33.743621,2.477292
4,EUSTX,EUSTX_TRANSFORMERS_DAILY_1,8.418200,49.512524,2.356013
5,NKY,NKY_LSTM_DAILY_1,23.496800,23.604320,3.953711
6,NKY,NKY_LSTM_DAILY_2,18.858300,23.842504,-0.641486
7,NKY,NKY_LSTM_NC_DAILY,20.789000,24.185866,1.975734
8,NKY,NKY_LSTM_NC_DAILY_2,20.035700,25.009846,1.964660
9,NKY,NKY_TRANSFORMERS_DAILY_1,20.650300,23.085310,0.726001



Approximate metrics under TC grid


...


Compact Sharpe by TC (bps)


TC (bps)                                0        5        10       20  \
Market Run                                                              
EUSTX  EUSTX_LSTM_DAILY_1         0.558723 0.487231 0.415739 0.272755   
       EUSTX_LSTM_DAILY_2         0.616685 0.526604 0.436522 0.256358   
       EUSTX_LSTM_NC_DAILY        0.488914 0.389604 0.290294 0.091673   
       EUSTX_LSTM_NC_DAILY_2      0.552999 0.471058 0.389117 0.225234   
       EUSTX_TRANSFORMERS_DAILY_1 0.559298 0.502817 0.446337 0.333376   
NKY    NKY_LSTM_DAILY_1           0.666979 0.525696 0.384412 0.101846   
       NKY_LSTM_DAILY_2           0.616901 0.487531 0.358161 0.099421   
       NKY_LSTM_NC_DAILY          0.524675 0.416207 0.307740 0.090806   
       NKY_LSTM_NC_DAILY_2        0.532303 0.425884 0.319466 0.106628   
       NKY_TRANSFORMERS_DAILY_1   0.559474 0.438298 0.317123 0.074773   
QQQ    QQQ_LSTM_DAILY_1           0.894271 0.813968 0.733665 0.573058   
       QQQ_LSTM_DAILY_2           0.913526 0.819375 0.725225 0.536924   
       QQQ_LSTM_NC_DAILY          0.901801 0.816791 0.731781 0.561761   
       QQQ_LSTM_NC_DAILY_2        0.900543 0.818240 0.735936 0.571329   
       QQQ_TRANSFORMERS_DAILY_1   0.895440 0.811652 0.727863 0.560285   

TC (bps)                                 50  
Market Run                                   
EUSTX  EUSTX_LSTM_DAILY_1         -0.156198  
       EUSTX_LSTM_DAILY_2         -0.284134  
       EUSTX_LSTM_NC_DAILY        -0.504188  
       EUSTX_LSTM_NC_DAILY_2      -0.266414  
       EUSTX_TRANSFORMERS_DAILY_1 -0.005507  
NKY    NKY_LSTM_DAILY_1           -0.745853  
       NKY_LSTM_DAILY_2           -0.676799  
       NKY_LSTM_NC_DAILY          -0.559997  
       NKY_LSTM_NC_DAILY_2        -0.531884  
       NKY_TRANSFORMERS_DAILY_1   -0.652279  
QQQ    QQQ_LSTM_DAILY_1            0.091238  
       QQQ_LSTM_DAILY_2           -0.027980  
       QQQ_LSTM_NC_DAILY           0.051700  
       QQQ_LSTM_NC_DAILY_2         0.077507  
       QQQ_TRANSFORMERS_DAILY_1    0.057552


Break-even TC summary (bps)


Break-even TC vs 0 return (bps)  \
Market Run                                                           
EUSTX  EUSTX_LSTM_DAILY_1                                39.075886   
       EUSTX_LSTM_DAILY_2                                34.229138   
       EUSTX_LSTM_NC_DAILY                               24.615513   
       EUSTX_LSTM_NC_DAILY_2                             33.743621   
       EUSTX_TRANSFORMERS_DAILY_1                        49.512524   
NKY    NKY_LSTM_DAILY_1                                  23.604320   
       NKY_LSTM_DAILY_2                                  23.842504   
       NKY_LSTM_NC_DAILY                                 24.185866   
       NKY_LSTM_NC_DAILY_2                               25.009846   
       NKY_TRANSFORMERS_DAILY_1                          23.085310   
QQQ    QQQ_LSTM_DAILY_1                                  55.680824   
       QQQ_LSTM_DAILY_2                                  48.514101   
       QQQ_LSTM_NC_DAILY                                 53.040844   
       QQQ_LSTM_NC_DAILY_2                               54.708631   
       QQQ_TRANSFORMERS_DAILY_1                          53.434364   

                                   Break-even TC vs benchmark mean (bps)  
Market Run                                                                
EUSTX  EUSTX_LSTM_DAILY_1                                       1.672892  
       EUSTX_LSTM_DAILY_2                                      -0.544854  
       EUSTX_LSTM_NC_DAILY                                     -0.731868  
       EUSTX_LSTM_NC_DAILY_2                                    2.477292  
       EUSTX_TRANSFORMERS_DAILY_1                               2.356013  
NKY    NKY_LSTM_DAILY_1                                         3.953711  
       NKY_LSTM_DAILY_2                                        -0.641486  
       NKY_LSTM_NC_DAILY                                        1.975734  
       NKY_LSTM_NC_DAILY_2                                      1.964660  
       NKY_TRANSFORMERS_DAILY_1                                 0.726001  
QQQ    QQQ_LSTM_DAILY_1                                        -0.926642  
       QQQ_LSTM_DAILY_2                                        -7.409251  
       QQQ_LSTM_NC_DAILY                                        5.252322  
       QQQ_LSTM_NC_DAILY_2                                      1.930573  
       QQQ_TRANSFORMERS_DAILY_1                                -1.322103


Approximation note:
- Net returns are shifted by ADT-based linear TC adjustment; daily turnover path is not available in saved RL outputs.
- Use this for sensitivity direction and rough break-even levels, not as exact pathwise recomputation.
